# PDF 파서 파이프라인
특허/논문 PDF → Markdown 변환 + Claude Vision API 수식 보완

In [ ]:
# 셀 1 — 환경 세팅 (세션마다 실행)
import os
!apt-get install -y openjdk-17-jdk-headless -qq
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
!pip install "opendataloader-pdf[hybrid]" pymupdf anthropic -q
!java -version
print("✓ 완료")

In [ ]:
# 셀 2 — PDF 업로드 & 파싱
from google.colab import files
import opendataloader_pdf
import os

uploaded = files.upload()  # Ctrl+클릭으로 다중 선택 가능

# 파일명 공백/괄호 제거 (CLI 파싱 오류 방지)
pdf_files = []
for fname in uploaded.keys():
    clean = fname.replace(" ", "_").replace("(", "").replace(")", "")
    if fname != clean:
        os.rename(fname, clean)
    pdf_files.append(clean)

print(f"업로드된 파일 {len(pdf_files)}개: {pdf_files}")

opendataloader_pdf.convert(
    input_path=pdf_files,
    output_dir="./output",
    format="markdown,json"
    # hybrid 미사용: 수식 개선 효과 없고 느리기만 함
)

In [ ]:
# 셀 3 — JSON 구조 확인 (셀 4 실행 전 반드시 확인)
import json, os

# pdf_files[0] 기준 자동 경로 설정
base_name = os.path.splitext(pdf_files[0])[0]
JSON_PATH = f"./output/{base_name}.json"

with open(JSON_PATH) as f:
    data = json.load(f)

# 최상위 키 확인
print("=== 최상위 키 ===")
print(list(data.keys()))

# elements 키 존재 여부 및 첫 5개 확인
elements = data.get("elements", data.get("blocks", data.get("content", [])))
print(f"\n=== elements 후보 키로 찾은 항목 수: {len(elements)} ===")

if elements:
    print("\n=== 첫 번째 element 구조 ===")
    print(json.dumps(elements[0], indent=2, ensure_ascii=False))

    # 실제 사용된 키 집합 확인
    all_keys = set()
    all_types = set()
    for el in elements:
        all_keys.update(el.keys())
        if "type" in el:
            all_types.add(el["type"])
    print(f"\n=== element에 쓰인 모든 키: {sorted(all_keys)} ===")
    print(f"=== type 값 종류: {sorted(all_types)} ===")
else:
    print("\n[주의] elements 키를 찾지 못했습니다. 전체 구조를 확인하세요:")
    print(json.dumps(data, indent=2, ensure_ascii=False)[:3000])

In [ ]:
# 셀 4 — 수식 보완 (Claude Vision API)
# 셀 3 실행 후 ELEMENTS_KEY / TYPE_KEY / PAGE_KEY / BBOX_KEY 를 실제 키로 수정할 것
import json, fitz, anthropic, base64, re, os

API_KEY   = "sk-ant-..."  # TODO: API 키 입력
PDF_PATH  = pdf_files[0]
JSON_PATH = f"./output/{os.path.splitext(pdf_files[0])[0]}.json"
MD_PATH   = f"./output/{os.path.splitext(pdf_files[0])[0]}.md"

# ↓ 셀 3 결과에 맞게 수정
ELEMENTS_KEY = "elements"  # 최상위에서 element 목록을 담는 키
TYPE_KEY     = "type"      # element 내 타입 키
PARA_TYPE    = "paragraph" # 단락을 나타내는 type 값
TEXT_KEY     = "text"      # 텍스트 내용 키
PAGE_KEY     = "page"      # 페이지 번호 키
BBOX_KEY     = "bbox"      # 바운딩박스 키 ([x0,y0,x1,y1])

with open(JSON_PATH) as f:
    data = json.load(f)

client = anthropic.Anthropic(api_key=API_KEY)
doc    = fitz.open(PDF_PATH)

def crop_to_base64(page_num, bbox):
    page = doc[page_num - 1]
    rect = fitz.Rect(bbox[0], bbox[1], bbox[2], bbox[3])
    clip = page.get_pixmap(matrix=fitz.Matrix(2, 2), clip=rect)
    return base64.standard_b64encode(clip.tobytes("png")).decode()

def to_latex(img_b64):
    resp = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=500,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": img_b64}},
                {"type": "text", "text": "이 이미지의 수식을 LaTeX로 변환해줘. $$ $$ 블록으로 감싸서 수식만 출력해. 설명 없이."}
            ]
        }]
    )
    return resp.content[0].text.strip()

elements  = data.get(ELEMENTS_KEY, [])
latex_map = {}

for el in elements:
    if el.get(TYPE_KEY, "") != PARA_TYPE:
        continue
    text = el.get(TEXT_KEY, "")
    if len(text) < 100 and re.search(r'[¼½¾∈ℜ∑∂⁎αβγηΦ]|argm|ð\d\)', text):
        page = el.get(PAGE_KEY, 1)
        bbox = el.get(BBOX_KEY, [])
        if not bbox:
            continue
        img_b64 = crop_to_base64(page, bbox)
        latex   = to_latex(img_b64)
        latex_map[(page, tuple(bbox))] = (text, latex)
        print(f"p.{page}: {text[:50]} → {latex[:60]}")

doc.close()
print(f"총 {len(latex_map)}개 수식 변환 완료")

In [ ]:
# 셀 5 — MD 교체 & 다운로드
from google.colab import files

with open(MD_PATH) as f:
    md = f.read()

for (page, bbox), (original, latex) in latex_map.items():
    if original in md:
        md = md.replace(original, latex)

out_path = MD_PATH.replace(".md", "_fixed.md")
with open(out_path, "w") as f:
    f.write(md)

print(f"저장 완료: {out_path}")
files.download(out_path)